# Chapter 1 — Reading data · Notebook to complete

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch01_a_completer.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

The cells you need to edit are marked **TO COMPLETE**. Run the cells in order, from top to bottom: as long as an answer is missing, the check cells say so without raising an error. The solution is in `ch01_corrige.ipynb`, on the same companion page.

**Convention.** As in the book, the variance, the standard deviation, the z-score and the covariance divide by $n$ (`ddof=0`), unless stated otherwise.

**Contents.** Data · The lesson calculations · Exercises 1.1 to 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# The book's colors: one color = one role (observed, predicted, deviation, colleague's line)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Data

The identifiers (D0.1, D1.1…) are those of the book and of the CSV files.

In [ ]:
# Chapter 1 data, copied from the book's CSV files (do not modify)

# D0.1 — the six homes of the opening
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — sale prices of the T2 apartments (two rooms, k€), lesson 1.1
north = [174, 177, 181, 183, 185]
south  = [150, 165, 185, 195, 205]

# D1.2 — register of sales in the South district, lessons 1.2 to 1.5
sales_log = pd.DataFrame({
    'sale':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'rooms':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'floor_area':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'energy_class':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'price':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'days_to_sell': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — price per m² by floor (k€/m²), lesson 1.5
floors = pd.DataFrame({
    'floor':        [0, 1, 2, 3, 4, 5, 6],
    'price_per_m2': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — selling times of three agencies (days), exercise 1.1
agency_delays = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — monthly rents of the studios (€), exercise 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `home` | home identifier (A, B…) | — | `logement` |
| `floor_area` | living area of the home | m² | `surface_m2` |
| `price` | sale price | k€ | `prix_kEUR` |
| `sale` | sale identifier (S01…) | — | `vente` |
| `rooms` | number of rooms | rooms | `pieces` |
| `energy_class` | energy performance rating class (A to G) | — | `dpe` |
| `days_to_sell` | time to sell | days | `delai_jours` |
| `floor` | floor | — | `etage` |
| `price_per_m2` | price per square meter | k€/m² | `prix_m2_kEUR` |

In [ ]:
# Tools provided: you do not need to modify them

def book_quartiles(values):
    """Q1 and Q3 by the book's rule (equation 1.5): ranks ⌈n/4⌉ and ⌈3n/4⌉."""
    x = sorted(values)
    n = len(x)
    rank_q1 = -(-n // 4)          # n/4 rounded up to the next integer
    rank_q3 = -(-3 * n // 4)      # 3n/4 rounded up to the next integer
    return x[rank_q1 - 1], x[rank_q3 - 1]   # rank 1 = position 0 in Python


def dot_rows(series, xlabel, bands=None, ax=None):
    """One row of dots per series, as in the figures of lesson 1.1.
    bands: dictionary {name: (mean, standard deviation)} to draw mean ± standard deviation."""
    if ax is None:
        fig, ax = plt.subplots()
    names = list(series)
    colors = [BLUE, ORANGE, '#52514e']
    for i, name in enumerate(names):
        y = len(names) - 1 - i
        if bands and name in bands:
            m, s = bands[name]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[name], [y] * len(series[name]), s=50,
                   color=colors[i % 3], zorder=3)
    ax.set_yticks(range(len(names)), list(reversed(names)))
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## The lesson calculations, in Python

These cells redo the calculations of lessons 1.1 to 1.5. Run them and compare with the book. Here, `np.std` and `np.var` are called with `ddof=0`, that is, with a division by $n$, as in the chapter.

In [ ]:
# Lesson 1.1 — mean, variance and standard deviation (division by n)
for name, one_series in [('North', north), ('South', south)]:
    print(f'{name}: mean = {np.mean(one_series):g} k€, variance = {np.var(one_series, ddof=0):g} k€², '
          f'standard deviation = {np.std(one_series, ddof=0):g} k€')

In [ ]:
# Lesson 1.2 — counts per energy rating class (ordinal qualitative variable)
sales_log['energy_class'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lesson 1.3 — mean, median and quartiles of the selling times
d = sales_log['days_to_sell']
print('mean:', d.mean(), ' median:', d.median())
print('without S13: mean', d[sales_log['sale'] != 'S13'].mean(),
      ' median', d[sales_log['sale'] != 'S13'].median())
print("quartiles, the book's rule (equation 1.5):", book_quartiles(d))
print('quartiles, NumPy default (interpolation):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lesson 1.4 — standardized scores (equation 1.6, standard deviation with division by n)
for name, one_series, sale in [('North', north, 188), ('South', south, 210)]:
    m, s = np.mean(one_series), np.std(one_series, ddof=0)
    print(f'{name}: z of the series = {np.round((np.array(one_series) - m) / s, 4)}, '
          f'z of the sale at {sale} k€ = {(sale - m) / s:g}')

In [ ]:
# Lesson 1.5 — covariance and correlation of the five T2 apartments in the South (division by n)
t2 = sales_log[sales_log['rooms'] == 2]
x, y = t2['floor_area'].to_numpy(float), t2['price'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('covariance:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r from np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r floor / price per m²:', np.corrcoef(floors['floor'], floors['price_per_m2'])[0, 1].round(4))

## Exercise 1.1 — Reading a chart

*Data D1.E1. Concepts: mean and standard deviation (lesson 1.1).* The chart below reproduces the one in exercise 1.1 of the book.

In [ ]:
dot_rows({f'Agency {a}': v for a, v in agency_delays.items()},
                  'Selling time (days)')
plt.xlim(0, 60)
plt.show()

**a) and b)** Estimate by eye the mean selling time of each agency, then rank the agencies from the least spread out to the most spread out.

In [ ]:
# TO COMPLETE: your estimates by eye, before any calculation
estimated_mean = {'P': None, 'Q': None, 'R': None}   # in days
spread_order = None   # list, from least to most spread out, e.g. ['Q', 'P', 'R']

In [ ]:
# Check (division by n)
for a, v in agency_delays.items():
    print(f'Agency {a}: mean = {np.mean(v):g} d, standard deviation = {np.std(v, ddof=0):.2f} d')
true_order = sorted(agency_delays, key=lambda a: np.std(agency_delays[a]))
if spread_order is None or None in estimated_mean.values():
    print('Complete the previous cell.')
else:
    print('Spread order:', 'correct' if spread_order == true_order else 'check again')

**c)** For Q and for R, count the sales closed within 2 days or less of the mean.

In [ ]:
# TO COMPLETE: your counts, read from the chart
close_to_Q = None
close_to_R = None

In [ ]:
# Check
for a, yours in [('Q', close_to_Q), ('R', close_to_R)]:
    v = np.array(agency_delays[a])
    truth = int(np.sum(np.abs(v - v.mean()) <= 2))
    if yours is None:
        print(f'Agency {a}: not yet completed.')
    else:
        print(f"Agency {a}: {'correct' if yours == truth else 'check again'}")

*Your explanation for c):* …

## Exercise 1.2 — Two residences, the same mean rent

*Data D1.E2. Concepts: mean, variance and standard deviation (lesson 1.1).* Camille cannot spend more than 570 € a month. Questions a) to d) are first done on paper, with the table of exercise 1.2 in the book. This notebook lets you check your answers, then program the standard deviation (question e) and compare two conventions (question f).

In [ ]:
dot_rows({'Les Tilleuls': tilleuls, 'Le Port': port}, "Monthly rent of a studio (€)")
plt.axvline(570, color='black', linestyle=':', label="Camille's budget: 570 €")
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Reading the rents

How many studios can Camille rent in each residence with 570 €, then with 620 €?

In [ ]:
# TO COMPLETE: number of affordable studios, read from the list of rents
affordable = {
    570: {'Les Tilleuls': None, 'Le Port': None},
    620: {'Les Tilleuls': None, 'Le Port': None},
}

In [ ]:
# Check: `<=` produces true/false values, np.sum counts the "true" ones
for budget, answers in affordable.items():
    for name, one_series in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        truth = int(np.sum(np.array(one_series) <= budget))
        yours = answers[name]
        status = 'not yet completed' if yours is None else ('correct' if yours == truth else 'check again')
        print(f'Budget {budget} €, {name}: {status}')

*Your answer to a), in one sentence:* …

### b) The two means

Compute them on paper, then enter your results below.

In [ ]:
# TO COMPLETE: your means, computed by hand (in €)
mean_tilleuls_by_hand = None
mean_port_by_hand = None

In [ ]:
# Check
for name, one_series, yours in [('Les Tilleuls', tilleuls, mean_tilleuls_by_hand),
                              ('Le Port', port, mean_port_by_hand)]:
    if yours is None:
        print(f'{name}: complete the previous cell.')
    else:
        ok = np.isclose(yours, np.mean(one_series))
        print(f"{name}: {'correct' if ok else 'check again'} (your value: {yours} €)")

### c) Variance and standard deviation by hand (division by $n = 6$)

On paper, draw up for each residence the table of deviations from the mean and of their squares (model: the table of the step-by-step calculation in lesson 1.1). Enter the four results here.

In [ ]:
# TO COMPLETE: your results, computed by hand (division by n)
variance_tilleuls_by_hand = None   # in €²
variance_port_by_hand = None       # in €²
std_tilleuls_by_hand = None # in €
std_port_by_hand = None     # in €

In [ ]:
# Check (division by n: ddof=0)
your_values = {
    'variance Tilleuls': (variance_tilleuls_by_hand, np.var(tilleuls, ddof=0)),
    'variance Port': (variance_port_by_hand, np.var(port, ddof=0)),
    'standard deviation Tilleuls': (std_tilleuls_by_hand, np.std(tilleuls, ddof=0)),
    'standard deviation Port': (std_port_by_hand, np.std(port, ddof=0)),
}
for name, (yours, expected) in your_values.items():
    if yours is None:
        print(f'{name}: not yet completed.')
    else:
        print(f"{name}: {'correct' if np.isclose(yours, expected) else 'check again'}")

### d) Interpreting

*Your answer:* …

### e) Programming the standard deviation

Complete the function by following equation 1.3 of the book, with a division by $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Use neither `np.std` nor `np.var`: the goal is to write each step yourself.

In [ ]:
# Exercise 1.2 e) — TO COMPLETE
def std_dev(values):
    """Standard deviation, division by n (equation 1.3)."""
    x = np.array(values, dtype=float)
    n = len(x)
    mean = None   # TO COMPLETE: equation 1.1
    deviations = None    # TO COMPLETE: one deviation per value
    variance = None  # TO COMPLETE: mean of the squares
    return None      # TO COMPLETE: back to the original unit

<details><summary><b>Hint</b> (click if you get stuck)</summary>

With NumPy, an operation written on an array applies to every value: `x - mean` gives the six deviations at once, and `deviations ** 2` their six squares. `np.sum(...)` adds up, `np.sqrt(...)` takes the square root.

</details>

In [ ]:
# Check: run this cell without modifying it
if std_dev(tilleuls) is None:
    print('The function still returns None: complete it.')
else:
    for name, one_series, expected in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtained = std_dev(one_series)
        status = 'correct' if np.isclose(obtained, expected) else f'check again (expected: {expected})'
        print(f'{name}: {obtained:.2f} € → {status}')

### f) Going further: dividing by $n$ or by $n - 1$

Run the cell below. Find 10.95 by hand by dividing the sum of squares by $n - 1 = 5$. Which pandas option gives the chapter's convention?

*Your answer:* …

In [ ]:
rents = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, default (division by n − 1):', rents.std().round(2).to_dict())
print('pandas, ddof=0 (division by n)         :', rents.std(ddof=0).round(2).to_dict())

## Exercise 1.3 — A surprising sale?

*Data D1.1. Concepts: standardized score (lesson 1.4, equation 1.6).* A T2 (two-room apartment) has just sold for 196 k€. Means and standard deviations: those of lesson 1.1, with division by $n$.

In [ ]:
# TO COMPLETE: your calculations, done on paper
z_north = None
z_south = None
south_price_as_unusual = None   # in k€, question c)

In [ ]:
# Check (standard deviation with division by n)
expected_values = {
    'z North': (z_north, (196 - np.mean(north)) / np.std(north, ddof=0)),
    'z South': (z_south, (196 - np.mean(south)) / np.std(south, ddof=0)),
    'equivalent South price': (south_price_as_unusual,
                            np.mean(south) + (196 - np.mean(north)) / np.std(north, ddof=0) * np.std(south, ddof=0)),
}
for name, (yours, truth) in expected_values.items():
    status = 'not yet completed' if yours is None else ('correct' if np.isclose(yours, truth) else 'check again')
    print(f'{name}: {status}')

*Your answer to b):* …

## Exercise 1.4 — Do floor area and price go together?

*Data D0.1 (the six homes of the opening). Concepts: covariance and correlation (lesson 1.5, equations 1.7 and 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(homes['floor_area'], homes['price'], color=BLUE)
for _, l in homes.iterrows():
    plt.annotate(l['home'], (l['floor_area'], l['price']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Floor area (m²)')
plt.ylabel('Sale price (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Before any calculation: will $r$ be positive or negative? Close to 0, to 0.5 or to 1?

*Your answer:* …

**b)** Complete the lines below. Use a division by $n$ everywhere.

In [ ]:
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
n = len(x)

products = None     # TO COMPLETE: (x_i − x̄)(y_i − ȳ) for each home
covariance = None   # TO COMPLETE: equation 1.7, division by n
r = None            # TO COMPLETE: equation 1.8 (standard deviations with ddof=0)

<details><summary><b>Hint</b></summary>

`x.mean()` gives the mean, `x - x.mean()` the six deviations. The product `*` of two arrays multiplies element by element. `x.std(ddof=0)` gives the standard deviation with division by $n$.

</details>

In [ ]:
# Check
if r is None or covariance is None:
    print('Complete the previous cell.')
else:
    print('covariance:', round(covariance, 2), 'm²·k€ →',
          'correct' if np.isclose(covariance, 1637.5) else 'check again (expected: 1637.5)')
    print('r:', round(r, 4), '→', 'correct' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'check again')

**c)** A correlation close to 1, yet deviations of 5 to 11 k€ in absolute value: is this contradictory?

*Your answer:* …

In [ ]:
print('np.cov by default (division by n − 1):', np.cov(x, y)[0, 1])
print('np.cov with bias=True (division by n) :', np.cov(x, y, bias=True)[0, 1])

## Exercise 1.5 — Predict, then check

*Data D1.1. Concepts: mean and standard deviation (lesson 1.1), median and interquartile range (lesson 1.3).* A sixth T2 sells in the North for 240 k€.

**a)** Before any calculation, write down your predictions: do the mean, the median, the standard deviation and the interquartile range increase a lot, a little or not at all?

*Your predictions:* …

In [ ]:
north_plus = north + [240]
print('North:', sorted(north))
print('North with the sale at 240 k€:', sorted(north_plus))

**b)** Complete the function `summary`. It must return the four indicators, with the standard deviation divided by $n$ and the quartiles computed by the book's rule (`book_quartiles`, provided above). The dictionary keys are `mean`, `median`, `std_dev` (standard deviation) and `iqr` (interquartile range).

In [ ]:
# TO COMPLETE
def summary(one_series):
    q1, q3 = book_quartiles(one_series)
    return {
        'mean': None,              # TO COMPLETE
        'median': None,              # TO COMPLETE: np.median
        'std_dev': None,           # TO COMPLETE: division by n
        'iqr': None,  # TO COMPLETE: from q1 and q3
    }

In [ ]:
table = pd.DataFrame({'North (5 sales)': summary(north), 'North+240 k€ (6 sales)': summary(north_plus),
                        'South (5 sales)': summary(south)})
if table.isna().any().any():
    print('Complete the function summary.')
else:
    expected = {'mean': (180, 190), 'median': (181, 182), 'std_dev': (4, 22.66),
               'iqr': (6, 8)}
    for k, (before, after) in expected.items():
        ok = np.isclose(table.loc[k].iloc[0], before) and np.isclose(table.loc[k].iloc[1], after, atol=0.005)
        print(f"{k}: {'correct' if ok else 'check again'}")
table.round(2)

In [ ]:
# The two series, modeled on the "mean ± standard deviation" figure of lesson 1.1
if table.isna().any().any():
    print('Complete the function summary to display the chart.')
else:
    dot_rows({'North': north, 'North + 240': north_plus},
                      'Sale price of a T2 (k€)',
                      bands={'North': (np.mean(north), np.std(north, ddof=0)),
                              'North + 240': (np.mean(north_plus), np.std(north_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

**c)** Have prices in the North become less predictable than those in the South?

*Your answer:* …

## Exercise 1.6 — Challenge: two lines, the same mean error

*Data D0.1. Concepts: signed deviation and absolute value (opening, lesson 1.1), standard deviation (lesson 1.1).* Line drawn by eye: predicted price = 3 × floor area + 30. Lazy line: predicted price = 222 k€ for every home.

In [ ]:
price = homes['price'].to_numpy(dtype=float)
floor_area = homes['floor_area'].to_numpy(dtype=float)

eye_line_errors = None          # TO COMPLETE: observed − predicted, line drawn by eye
lazy_line_errors = None    # TO COMPLETE: observed − predicted, lazy line

In [ ]:
# The three indicators of question b) (standard deviation: division by n)
if eye_line_errors is None or lazy_line_errors is None:
    print('Complete the previous cell.')
else:
    indicators = pd.DataFrame({
        name: {'mean of signed deviations': e.mean(),
              'mean of absolute deviations': np.abs(e).mean(),
              'standard deviation of signed deviations': e.std(ddof=0)}
        for name, e in [('line by eye', eye_line_errors), ('lazy line', lazy_line_errors)]
    })
    print('deviations, line by eye:', eye_line_errors)
    print('deviations, lazy line :', lazy_line_errors)
    print('check:', 'correct' if np.allclose(indicators.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicators.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'check again')
    display(indicators.round(2))

In [ ]:
if eye_line_errors is not None and lazy_line_errors is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(floor_area - 0.8, eye_line_errors, color=ORANGE, label="line by eye")
    ax.scatter(floor_area + 0.8, lazy_line_errors, color=BLUE, marker='s', label='lazy line')
    ax.set_xlabel('Floor area (m²)')
    ax.set_ylabel('Deviation observed − predicted (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

*Your answers to c) and d):* …

In [ ]:
# Check of d): the standard deviation of the lazy line's deviations equals that of the prices
if lazy_line_errors is not None:
    print(round(lazy_line_errors.std(ddof=0), 4), round(price.std(ddof=0), 4))